# Expense Analyst Agent — Walkthrough

Run **top to bottom**. Need a `.env` with `AZURE_OPENAI_ENDPOINT` and `AZURE_OPENAI_API_KEY`.

1. **Install + imports** — packages, then SDK / PDF / Gradio imports.
2. **Azure setup** — load keys, disable tracing, wire the chat model.
3. **Tools** — PDF text, CSV parse/validate, spending summary by category.
4. **Agent** — `ExpenseAnalystAgent` uses those tools.
5. **Tests** — one PDF run, one CSV run. Check the printed summaries.
6. **Chat** — last code cell opens Gradio. Sample files: `data/MonthlySpendingData_Placeholder.pdf`, `data/sample_transactions.csv`.

## 1. Install packages

Run this once. Installs the Agents SDK, Azure OpenAI client, PDF reader, Gradio, and pandas.

In [16]:
!uv pip install --upgrade openai openai-agents pypdf python-dotenv requests gradio pandas

Resolved 78 packages in 509ms
Checked 78 packages in 26ms


## Imports

Brings in dotenv, Azure OpenAI, Agents SDK, PDF reader, Gradio, and pandas. No output expected.

In [17]:
from dotenv import load_dotenv
import os
from pydantic import BaseModel

from openai import AsyncAzureOpenAI
from agents import (
    Agent,
    Runner,
    function_tool,
    OpenAIChatCompletionsModel,
    set_tracing_disabled,
    SQLiteSession,
    GuardrailFunctionOutput,
    RunContextWrapper,
    InputGuardrailTripwireTriggered,
    OutputGuardrailTripwireTriggered,
)

from agents.decorators import input_guardrail, output_guardrail
from agents.extensions.handoff_prompt import RECOMMENDED_PROMPT_PREFIX
from pypdf import PdfReader
import gradio as gr
import pandas as pd

## 2. Load Azure keys

Reads `.env`. Both prints should be `True`. If not, stop and fix the env file.

In [18]:
load_dotenv(override=True)

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
DEPLOYMENT = "gpt-5.4-ptu"

print("Endpoint loaded:", bool(AZURE_OPENAI_ENDPOINT))
print("API key loaded:", bool(AZURE_OPENAI_API_KEY))

Endpoint loaded: True
API key loaded: True


## 3. Wire the Azure model

Turns tracing off, then builds `azure_model` for every agent call below.

In [19]:
set_tracing_disabled(True)

azure_client = AsyncAzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version="2024-10-21",
)

azure_model = OpenAIChatCompletionsModel(
    model=DEPLOYMENT,
    openai_client=azure_client,
)

print("Azure model configured for the OpenAI Agents SDK.")

Azure model configured for the OpenAI Agents SDK.


## 4. PDF tool

Pulls text from `data/MonthlySpendingData_Placeholder.pdf`. `extract_pdf_text` is the tool the agent can call. You should see a character-count print.

In [20]:
def read_pdf(file_path: str) -> str:
    reader = PdfReader(file_path)
    text = ""

    for page in reader.pages:
        page_text = page.extract_text()

        if page_text:
            text += page_text + "\n"

    return text


# AI Agent Tool
@function_tool
def extract_pdf_text(file_path: str) -> str:
    """Extract text content from a PDF file."""
    return read_pdf(file_path)


# PDF Collection
MONTHLY_EXPENSES_COLLECTION = {
    "monthly_expenses_pdf_1": {
        "filename": "MonthlySpendingData_Placeholder.pdf",
        "content": read_pdf(
            "data/MonthlySpendingData_Placeholder.pdf"
        )
    }
}


spending_data_text = MONTHLY_EXPENSES_COLLECTION[
    "monthly_expenses_pdf_1"
]["content"]

print("Loaded PDF text length:", len(spending_data_text), "characters")

Loaded PDF text length: 275 characters


## 5. CSV validation

Banks export CSVs differently — this file is a placeholder. Required columns: `date`, `description`, `amount`. Bad rows raise an error.

In [21]:
# Validate csv file
def load_validated_csv(file_path: str) -> pd.DataFrame:
    df = pd.read_csv(file_path)

    required = {"date", "description", "amount"}
    missing = required - set(df.columns)

    if missing:
        raise ValueError(f"Missing columns: {missing}")

    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df["amount"] = pd.to_numeric(df["amount"], errors="coerce")

    if df[["date", "amount", "description"]].isna().any().any():
        raise ValueError("CSV contains invalid or missing values")

    if df["description"].astype(str).str.strip().eq("").any():
        raise ValueError("CSV contains empty descriptions")

    return df

## 6. CSV parse tool

Wraps the validator and returns transactions as JSON. On a bad file it returns `success: False` plus the error.

In [22]:
@function_tool
def parse_csv_transactions_tool(file_path: str) -> dict:
    """Parse and validate transactions from a CSV file."""
    try:
        df = load_validated_csv(file_path)

        return {
            "success": True,
            "transaction_count": len(df),
            "transactions": df.assign(
                date=df["date"].dt.strftime("%Y-%m-%d")
            ).to_dict(orient="records")
        }

    except (
        OSError,
        ValueError,
        pd.errors.ParserError,
        UnicodeError
    ) as e:
        return {
            "success": False,
            "error": str(e)
        }

## 7. Spending summary tool

Totals spend and buckets merchants into Groceries / Dining / Transportation / Other.

In [23]:
@function_tool
def calculate_receipt_summary_tool(file_path: str) -> dict:
    """Read a grocery receipt CSV and return detailed item information and spending summary."""
    try:
        df = load_validated_csv(file_path)

        # Make sure the receipt has the required columns
        required_columns = ["item", "quantity", "price"]

        missing_columns = [
            column for column in required_columns
            if column not in df.columns
        ]

        if missing_columns:
            return {
                "success": False,
                "error": f"Missing required columns: {', '.join(missing_columns)}"
            }

        # Convert numeric fields
        df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")
        df["price"] = pd.to_numeric(df["price"], errors="coerce")

        if df["quantity"].isna().any() or df["price"].isna().any():
            return {
                "success": False,
                "error": "Quantity and price must contain valid numbers."
            }

        # Calculate total for each item
        df["item_total"] = (df["quantity"] * df["price"]).round(2)

        # Categorize grocery items
        def categorize_item(item: str) -> str:
            item = str(item).lower()

            if any(word in item for word in [
                "apple", "banana", "orange", "lettuce",
                "tomato", "potato", "onion", "vegetable",
                "fruit", "salad"
            ]):
                return "Produce"

            elif any(word in item for word in [
                "milk", "cheese", "yogurt", "butter",
                "cream", "egg"
            ]):
                return "Dairy"

            elif any(word in item for word in [
                "chicken", "beef", "pork", "steak",
                "sausage", "turkey", "meat"
            ]):
                return "Meat"

            elif any(word in item for word in [
                "bread", "bun", "bagel", "tortilla",
                "cereal", "rice", "pasta"
            ]):
                return "Grains"

            elif any(word in item for word in [
                "chips", "cookie", "candy", "chocolate",
                "cracker", "snack"
            ]):
                return "Snacks"

            elif any(word in item for word in [
                "soda", "juice", "water", "coffee",
                "tea", "drink"
            ]):
                return "Beverages"

            elif any(word in item for word in [
                "frozen", "ice cream"
            ]):
                return "Frozen Foods"

            else:
                return "Other"

        df["category"] = df["item"].apply(categorize_item)

        # Calculate spending by category
        spending_by_category = (
            df.groupby("category")["item_total"]
            .sum()
            .round(2)
            .to_dict()
        )

        # Create detailed item list
        items = []

        for _, row in df.iterrows():
            items.append({
                "item": row["item"],
                "quantity": int(row["quantity"]),
                "price_each": round(float(row["price"]), 2),
                "total": round(float(row["item_total"]), 2),
                "category": row["category"]
            })

        return {
            "success": True,
            "item_count": len(items),
            "total_before_tax": round(float(df["item_total"].sum()), 2),
            "spending_by_category": spending_by_category,
            "items": items
        }

    except (
        OSError,
        ValueError,
        pd.errors.ParserError,
        UnicodeError
    ) as e:
        return {
            "success": False,
            "error": str(e)
        }

## 8. Save tool (stub)

Placeholder only. Not attached to the agent yet — skip mentally, do not expect it to persist data.

In [24]:
@function_tool
def save_transactions_tool():
    return

## Guardrails

Input guardrail blocks off-topic messages before the agents run. Output guardrail blocks budget plans that give risky advice (specific investments, loans, tax/legal advice). Each guardrail is a small checker agent that returns a yes/no plus a reason.

### Topic checker agent

Classifies whether a message is about personal finance. Returns `TopicCheck` (structured output).

In [25]:
class TopicCheck(BaseModel):
    is_on_topic: bool
    reasoning: str


topic_checker_agent = Agent(
    name="TopicChecker",
    instructions=(
        "Decide if the user's message is about personal finance: expenses, spending, budgets, "
        "bills, subscriptions, saving money, grocery prices, or reading their expense files (CSV/PDF). "
        "Greetings and follow-up questions about their budget are on topic. "
        "Anything else (homework, coding, poems, general trivia) is off topic."
    ),
    output_type=TopicCheck,
    model=azure_model,
)

### Input guardrail: `finance_topic_guardrail`

Attached to OrchestratorAgent (the first agent the user talks to). `run_in_parallel=False` so the agent only runs after the check passes — no wasted tool calls on blocked messages.

In [26]:
@input_guardrail
async def finance_topic_guardrail(ctx: RunContextWrapper, agent: Agent, input) -> GuardrailFunctionOutput:
    result = await Runner.run(topic_checker_agent, input, context=ctx.context)
    check = result.final_output
    return GuardrailFunctionOutput(output_info=check, tripwire_triggered=not check.is_on_topic)

print("Guardrail function built:", finance_topic_guardrail.name)

Guardrail function built: finance_topic_guardrail


### Advice checker agent

Reviews a budget plan and flags risky advice. Returns `AdviceCheck` (structured output).

In [27]:
class AdviceCheck(BaseModel):
    gives_risky_advice: bool
    reasoning: str


advice_checker_agent = Agent(
    name="AdviceChecker",
    instructions=(
        "You review a budget plan written for a user. Flag it (gives_risky_advice = true) only if it "
        "recommends specific investments (named stocks, crypto, funds), taking loans or credit to save money, "
        "or tax/legal advice. Normal budgeting tips (cut dining 20%, cancel subscriptions, buy store brands, "
        "build an emergency fund) are fine."
    ),
    output_type=AdviceCheck,
    model=azure_model,
)

### Output guardrail: `safe_advice_guardrail`

Attached to OrchestratorAgent (the final answer the user sees).

In [28]:
@output_guardrail
async def safe_advice_guardrail(ctx: RunContextWrapper, agent: Agent, output: str) -> GuardrailFunctionOutput:
    result = await Runner.run(advice_checker_agent, output, context=ctx.context)
    check = result.final_output
    return GuardrailFunctionOutput(output_info=check, tripwire_triggered=check.gives_risky_advice)

print("Guardrail function built:", safe_advice_guardrail.name)

Guardrail function built: safe_advice_guardrail


## 9. Build ExpenseAnalystAgent

System prompt + three tools: PDF extract, CSV parse, spending summary. Print should say `Agent built: ExpenseAnalystAgent`.

In [29]:
instructions = f"""{RECOMMENDED_PROMPT_PREFIX}

You are Expense Analyst Agent, You are a Financial Advizing Agent that Analyzes financial data provided by users through 
CSV files, chat inputs, PDF files. You categorizes transactions, identifies spending patterns, and summarizes expenses for the user

#Responsibilities
You are to, when prompted by the user, to list the monthly expenses in the files provided, whether that be the pdf of csv files.
then you handoff the data to the BudgetOptimizerAgent. 
Explicitly state what part of the output was provided by the ExpenseAnalystAgent
Explicitly state which tools are used and when.


"""

ExpenseAnalystAgent = Agent(
    name="ExpenseAnalystAgent",
    handoff_description="Analyzes budget and summarizes spending.",
    instructions=instructions,
    tools=[extract_pdf_text, parse_csv_transactions_tool, calculate_receipt_summary_tool],
    model=azure_model,
)

print("Agent built:", ExpenseAnalystAgent.name)




Agent built: ExpenseAnalystAgent


## 10. Compare Grocery Prices tool

Tool used by Budget Optimizer Agent. Uses the free Kroger Public API: finds the nearest Kroger store by ZIP, searches a product, and ranks brands by unit price so the agent can suggest a cheaper brand of the same item. Needs `KROGER_CLIENT_ID` and `KROGER_CLIENT_SECRET` in `.env`.

### Kroger setup + token

Reads Kroger keys from `.env` and gets an OAuth2 token (cached until it expires).

In [30]:
import re
import time
import requests

KROGER_CLIENT_ID = os.getenv("KROGER_CLIENT_ID")
KROGER_CLIENT_SECRET = os.getenv("KROGER_CLIENT_SECRET")
# Production by default; set KROGER_BASE_URL=https://api-ce.kroger.com/v1 in .env for a Certification app
KROGER_BASE_URL = os.getenv("KROGER_BASE_URL", "https://api.kroger.com/v1")

_kroger_token = {"value": None, "expires_at": 0}


def get_kroger_token() -> str:
    """Get (and cache) an OAuth2 client-credentials token for the Kroger API."""
    if _kroger_token["value"] and time.time() < _kroger_token["expires_at"] - 60:
        return _kroger_token["value"]

    response = requests.post(
        f"{KROGER_BASE_URL}/connect/oauth2/token",
        auth=(KROGER_CLIENT_ID, KROGER_CLIENT_SECRET),
        data={"grant_type": "client_credentials", "scope": "product.compact"},
        timeout=10,
    )
    response.raise_for_status()
    body = response.json()
    _kroger_token["value"] = body["access_token"]
    _kroger_token["expires_at"] = time.time() + body["expires_in"]
    return _kroger_token["value"]

### Kroger request helper

Sends an authorized GET request to the Kroger API.

In [31]:
def kroger_get(path: str, params: dict) -> dict:
    response = requests.get(
        f"{KROGER_BASE_URL}{path}",
        headers={"Authorization": f"Bearer {get_kroger_token()}"},
        params=params,
        timeout=10,
    )
    response.raise_for_status()
    return response.json()

### Unit size parser

Turns sizes like `1 gal`, `1/2 gal`, `2 lb` into one base unit (fl oz, oz, ct) so different brands can be compared fairly.

In [32]:
# Convert a size string like "64 fl oz", "1 gal", "2 lb", "12 ct" into (amount, base_unit)
UNIT_CONVERSIONS = {
    "oz": (1, "oz"), "lb": (16, "oz"), "g": (0.035274, "oz"), "kg": (35.274, "oz"),
    "fl oz": (1, "fl oz"), "gal": (128, "fl oz"), "qt": (32, "fl oz"), "pt": (16, "fl oz"),
    "l": (33.814, "fl oz"), "ml": (0.033814, "fl oz"),
    "ct": (1, "ct"),
}


def parse_size(size: str):
    match = re.match(r"\s*(\d+(?:\.\d+)?)(?:/(\d+))?\s*(fl oz|oz|lb|gal|qt|pt|ml|kg|g|l|ct)\b", size.lower())
    if not match:
        return None
    amount = float(match.group(1)) / (float(match.group(2)) if match.group(2) else 1)
    factor, base_unit = UNIT_CONVERSIONS[match.group(3)]
    return amount * factor, base_unit

### Product summary

Keeps only priced products and builds rows: brand, size, price (sale price if lower), and unit price.

In [33]:
def summarize_products(products: list) -> list:
    """Flatten Kroger product results into brand / price / unit-price rows."""
    rows = []
    for product in products:
        item = (product.get("items") or [{}])[0]
        price_info = item.get("price") or {}
        regular = price_info.get("regular")
        if not regular:
            continue  # no price at this store
        promo = price_info.get("promo")
        price = promo if promo and promo < regular else regular
        size = item.get("size", "")
        parsed = parse_size(size)

        rows.append({
            "brand": product.get("brand", "Unknown"),
            "category": (product.get("categories") or ["Unknown"])[0],
            "description": product.get("description", ""),
            "size": size,
            "price": round(price, 2),
            "on_sale": price < regular,
            "unit_price": round(price / parsed[0], 3) if parsed and parsed[0] else None,
            "unit": parsed[1] if parsed else None,
        })
    return rows

### Saved price data (cache)

Every live Kroger lookup is saved to `data/kroger_price_cache.json`. Without a Kroger API key, the AI can still use the tool - it reads from this file instead.

In [34]:
import json
from datetime import date

KROGER_CACHE_PATH = "data/kroger_price_cache.json"


def load_price_cache() -> dict:
    if not os.path.exists(KROGER_CACHE_PATH):
        return {}
    with open(KROGER_CACHE_PATH) as f:
        return json.load(f)


def save_to_price_cache(product_name: str, entry: dict) -> None:
    cache = load_price_cache()
    cache[product_name.lower().strip()] = entry
    with open(KROGER_CACHE_PATH, "w") as f:
        json.dump(cache, f, indent=2)

### Live Kroger lookup

Nearest Kroger store by ZIP → search the product → summarized rows. Saves the result to the cache.

In [35]:
def fetch_kroger_prices(product_name: str, zip_code: str) -> dict:
    locations = kroger_get("/locations", {"filter.zipCode.near": zip_code, "filter.limit": 1})["data"]
    if not locations:
        raise ValueError(f"No Kroger store found near {zip_code}")
    store = locations[0]

    products = kroger_get("/products", {
        "filter.term": product_name,
        "filter.locationId": store["locationId"],
        "filter.limit": 20,
    })["data"]

    entry = {
        "store": f"{store['name']} ({store['address']['addressLine1']}, {store['address']['city']})",
        "zip_code": zip_code,
        "fetched_on": date.today().isoformat(),
        "rows": summarize_products(products),
    }
    save_to_price_cache(product_name, entry)
    return entry

### Rank brands

Keeps only products in the most common category (search also returns unrelated items), then compares by unit price (using the most common unit) so different sizes are fair; falls back to shelf price.

In [36]:
def rank_brands(rows: list) -> dict:
    # Keyword search also returns unrelated items (e.g. "peanut butter" -> Reese's candy),
    # so only compare products in the most common category
    categories = [r.get("category", "Unknown") for r in rows]
    main_category = max(set(categories), key=categories.count)
    rows = [r for r in rows if r.get("category", "Unknown") == main_category]

    units = [r["unit"] for r in rows if r["unit"]]
    main_unit = max(set(units), key=units.count) if units else None
    comparable = [r for r in rows if main_unit and r["unit"] == main_unit]

    if comparable:
        ranked, basis, key = sorted(comparable, key=lambda r: r["unit_price"]), f"price per {main_unit}", "unit_price"
    else:
        ranked, basis, key = sorted(rows, key=lambda r: r["price"]), "shelf price (sizes not comparable)", "price"

    cheapest, priciest = ranked[0], ranked[-1]
    savings_pct = round((1 - cheapest[key] / priciest[key]) * 100, 1) if priciest[key] else 0

    return {
        "category": main_category,
        "compared_by": basis,
        "cheapest_brand": cheapest,
        "most_expensive_brand": priciest,
        "savings_percentage": savings_pct,
        "all_options": ranked[:10],
    }

### `compare_grocery_prices_tool`

The tool the agent calls. With a Kroger key → live prices (and saves them). Without a key → saved prices from `data/kroger_price_cache.json`.

In [37]:
@function_tool
def compare_grocery_prices_tool(product_name: str, zip_code: str = "45221") -> dict:
    """Compare prices of the same grocery product across different brands at a Kroger store.
    Returns brands sorted from cheapest to most expensive per unit (e.g., per oz), so a cheaper brand can be suggested."""
    has_key = bool(KROGER_CLIENT_ID and KROGER_CLIENT_SECRET)

    if has_key:
        try:
            entry = fetch_kroger_prices(product_name, zip_code)
        except (requests.RequestException, KeyError, ValueError) as e:
            return {"success": False, "error": str(e)}
        source = "live Kroger API"
    else:
        entry = load_price_cache().get(product_name.lower().strip())
        if not entry:
            return {
                "success": False,
                "error": f"No Kroger API key and no saved prices for '{product_name}'.",
                "saved_products": sorted(load_price_cache()),
            }
        source = f"saved Kroger data from {entry['fetched_on']}"

    if not entry["rows"]:
        return {"success": False, "error": f"No priced results for '{product_name}' at {entry['store']}"}

    return {
        "success": True,
        "source": source,
        "store": entry["store"],
        "product": product_name,
        **rank_brands(entry["rows"]),
    }

### Build saved price data (only with a Kroger key)

Run once with a Kroger key to save prices for common groceries, then commit `data/kroger_price_cache.json`. Without a key this cell just skips.

In [38]:
COMMON_GROCERIES = [
    "milk", "eggs", "bread", "butter", "cheese", "yogurt", "white rice", "pasta", "cereal",
    "chicken breast", "ground beef", "ground coffee", "peanut butter", "orange juice",
]

if KROGER_CLIENT_ID and KROGER_CLIENT_SECRET:
    for item in COMMON_GROCERIES:
        try:
            # 45221 is Cincinnati zipcode
            entry = fetch_kroger_prices(item, "45221")
            print(f"Saved {item}: {len(entry['rows'])} priced products")
        except (requests.RequestException, KeyError, ValueError) as e:
            print(f"Failed {item}: {e}")
else:
    print("No Kroger key — skipping. Saved products:", sorted(load_price_cache()))

No Kroger key — skipping. Saved products: ['bread', 'butter', 'cereal', 'cheese', 'chicken breast', 'eggs', 'ground beef', 'ground coffee', 'milk', 'orange juice', 'pasta', 'peanut butter', 'white rice', 'yogurt']


## 11. Identify Recurring Expenses Tool

Tool used by Budget Optimizer

In [39]:
## identify_recurring_expenses_Tool
@function_tool
def identify_recurring_expenses_Tool(Expenses_category: str)-> dict:
     """Locate Recurring expenses in database to see what the person recurrently spends on(e.g., Food, Water, tools)."""
     Expenses_tiers = {
        "Food":{
            "Quantity_Size" "Standard Supermarket (Kroger/Meijer)"
        },
        "Water":{
            "Quality_rating" "Local Gas stations (Speedway/Marathon)"
        },
        "Tools": {
            "Customer_Rating" "Hardware stores(Harbor Freight/Home)"
        }
    }

     if Expenses_category in Expenses_tiers:
        return  {"success": True, "results": Expenses_tiers[Expenses_category]}  
     return {"success": False, "message": "No expenses are recurring in this category"}

## 12. Calculate Budget Adjustments tool

Tool used by Budget Optimizer. Estimates how much the user could save by reducing spending in one category: new monthly budget plus monthly and annual savings.

In [40]:
## calculate_budget_adjustments_tool
@function_tool
def calculate_budget_adjustments_tool(category: str, current_monthly_amount: float, reduction_rate: float) -> dict:
    """Calculate the new monthly budget and monthly/annual savings for one category.
    reduction_rate is a fraction between 0 and 1 (e.g. 0.20 = cut 20%)."""
    if current_monthly_amount < 0:
        return {"success": False, "error": "current_monthly_amount must be >= 0"}
    if not 0 <= reduction_rate <= 1:
        return {"success": False, "error": "reduction_rate must be between 0 and 1"}

    monthly_savings = round(current_monthly_amount * reduction_rate, 2)

    return {
        "success": True,
        "category": category,
        "current_monthly_amount": round(current_monthly_amount, 2),
        "reduction_rate": reduction_rate,
        "new_monthly_budget": round(current_monthly_amount - monthly_savings, 2),
        "monthly_savings": monthly_savings,
        "annual_savings": round(monthly_savings * 12, 2),
    }

## Agent 2: Build Budget Optimizer
System prompt + three tools: PDF extract, CSV parse, Budget Plan. Print should say `Agent built: Budget Optimizer`.

In [41]:
instructions = f"""{RECOMMENDED_PROMPT_PREFIX}

You are the Budget Optimizer Agent, a specialized financial advisor within a multi-agent personal finance service.
Your primary role is to take the structured expense and spending summary provided by the Expense Analyst Agent and identify unnecessary spending, recurring subscription leaks, and high-cost categories.

## responibilities
Analyze Category Breakdown: Examine spending across categories like Groceries, Dining, Transportation, Utilities, Other.
Detect Recurring Leaks: Use your tools to flag subscriptions, recurring bills, and minor purchases.
Compare Alternatives: The spending summary only shows grocery totals, not individual items. If Groceries spending appears, call compare_grocery_prices_tool for 2-3 common staples (e.g., milk, eggs, bread) to show the user a cheaper brand of the same item (compared by unit price). If the user names a specific product, compare that product instead. If the tool fails and returns saved_products, retry with items from that list. Never call the tool with a category name like "Groceries".
Calculate Adjustments: Use calculate_budget_adjustments_tool to compute the new budget and potential monthly and annual savings for each category you suggest reducing. Provide concrete, actionable budget adjustments.

##Rules
Do not make up numbers, base your calculations on the data provided by the Expense Analyst or retrieved via your tools.
Be constructive and realistic: Avoid shaming the user. Provide practical steps like reducing dining out by 20% rather than cutting it to zero.
Keep the answer short: under 350 words. Layout: one line with current spending and the new target; a small markdown table for price comparisons (item, cheapest, most expensive, savings %); 3-5 action bullets, one line each; one short follow-up question.
Show each number only once. Do not add a separate summary or conclusion section. Bold dollar amounts.
End with one line: "Provided by BudgetOptimizerAgent. Tools used: ..." listing the tools you called.
"""
BudgetOptimizerAgent = Agent(
    name="BudgetOptimizerAgent",
    handoff_description="Create and print a layout plan for spending.",
    instructions=instructions,
    tools=[compare_grocery_prices_tool, identify_recurring_expenses_Tool, calculate_budget_adjustments_tool],
    model=azure_model,
)

print("Agent built:", BudgetOptimizerAgent.name)


Agent built: BudgetOptimizerAgent


## Agent 3: Savings Planner

Helps the user set savings goals and builds a monthly savings plan from their income, expenses, and the Budget Optimizer's savings. Goals must persist across sessions, so they are stored as structured data in SQLite (not chat memory).

The three tool cells below are empty placeholders to be built.

### `save_savings_goal_tool`

Stores or updates one savings goal in SQLite so it is remembered in future sessions.

In [ ]:
## save_savings_goal_tool
@function_tool
def save_savings_goal_tool(goal_name: str, target_amount: float, target_date: str) -> dict:
    """Save a savings goal with a name, target amount, and target date."""
    if target_amount <= 0:
        return {"success": False, "error": "target_amount must be greater than 0"}
    try:
        target_date_parsed = pd.to_datetime(target_date)
    except ValueError:
        return {"success": False, "error": "target_date must be in YYYY-MM-DD format"}

    # Here you would normally save the goal to a database or file
    # For this example, we'll just return the goal details
    return {
        "success": True,
        "goal_name": goal_name,
        "target_amount": round(target_amount, 2),
        "target_date": target_date_parsed.strftime("%Y-%m-%d"),
        "message": f"Savings goal '{goal_name}' of **${round(target_amount, 2)}** by {target_date_parsed.strftime('%Y-%m-%d')} saved successfully."
    }

### `get_savings_goals_tool`

Reads the user's previously saved goals back from SQLite.

In [ ]:
## get_savings_goals_tool

### `calculate_savings_plan_tool`

Works out how much the user needs to save each month to hit a goal by its date, and whether that fits their budget.

In [ ]:
## calculate_savings_plan_tool

### Build SavingsPlannerAgent

System prompt + the three tools above. Not attached to the Orchestrator yet. Print should say `Agent built: SavingsPlannerAgent`.

In [ ]:
instructions = f"""{RECOMMENDED_PROMPT_PREFIX}

You are the Savings Planner Agent, the third specialist in a multi-agent personal finance service.
You help the user set realistic savings goals and build a monthly savings plan based on their income,
their expenses from the Expense Analyst Agent, and the savings found by the Budget Optimizer Agent.

## Responsibilities
Check saved goals: call get_savings_goals_tool first so you build on goals the user already set.
Save goals: when the user states a new goal or changes one, call save_savings_goal_tool.
Plan: call calculate_savings_plan_tool to get the monthly amount needed and whether it fits the budget.

## Rules
Do not make up numbers. If income or a target date is missing, ask the user for it.
If a goal does not fit the budget, suggest a later date or a smaller monthly amount instead.
End with one line: "Provided by SavingsPlannerAgent. Tools used: ..." listing the tools you called.
"""

SavingsPlannerAgent = Agent(
    name="SavingsPlannerAgent",
    handoff_description="Sets savings goals and builds a monthly savings plan.",
    instructions=instructions,
    tools=[],  # TODO: add save_savings_goal_tool, get_savings_goals_tool, calculate_savings_plan_tool once built
    model=azure_model,
)

print("Agent built:", SavingsPlannerAgent.name)

## Orchestrator
OrchestratorAgent decides which specialist to call. It always calls ExpenseAnalystAgent for spending data, and calls BudgetOptimizerAgent only when the user wants to save money or get a budget plan. Both specialists are attached as tools (`as_tool`), and both guardrails live on the orchestrator. Print should say `Agent built: OrchestratorAgent`.

In [42]:
orchestrator_instructions = """
You are the Orchestrator of a personal finance assistant. You do not analyze data yourself.
You decide which specialist agent to call:

- expense_analyst: reads the user's CSV/PDF files and summarizes spending by category.
  Call it first whenever the user's question needs their spending data. Pass the file path if the user gave one.
- budget_optimizer: finds spending leaks, compares grocery prices, and calculates a reduced budget.
  Call it ONLY when the user wants to save money, cut spending, or get a budget plan.
  Pass it the full summary returned by expense_analyst.

Rules:
- If the user only asks about their current spending, call expense_analyst only.
- Return the last specialist's answer as-is. Do not rewrite, summarize, or add sections.
- If no file was given and none is in the conversation, ask the user for a CSV or PDF file.
"""

OrchestratorAgent = Agent(
    name="OrchestratorAgent",
    instructions=orchestrator_instructions,
    tools=[
        ExpenseAnalystAgent.as_tool(
            tool_name="expense_analyst",
            tool_description="Read the user's expense file (CSV/PDF) and summarize spending by category.",
        ),
        BudgetOptimizerAgent.as_tool(
            tool_name="budget_optimizer",
            tool_description="Given a spending summary, find leaks, compare grocery prices, and build a budget plan.",
        ),
    ],
    input_guardrails=[finance_topic_guardrail],
    output_guardrails=[safe_advice_guardrail],
    model=azure_model,
)

print("Agent built:", OrchestratorAgent.name)

Agent built: OrchestratorAgent


In [43]:
chat_session = SQLiteSession("gradio_user", "user_expenses.db")

async def chained_gradio_chat(message, history):
    try:
        result = await Runner.run(OrchestratorAgent, message, session=chat_session)
        return result.final_output

    except InputGuardrailTripwireTriggered as e:
        reason = e.guardrail_result.output.output_info.reasoning
        return f"Sorry, I can only help with personal finance and budgeting questions.\n\n(Guardrail: {reason})"

    except OutputGuardrailTripwireTriggered as e:
        reason = e.guardrail_result.output.output_info.reasoning
        return f"I generated a plan that included advice I'm not allowed to give, so I blocked it.\n\n(Guardrail: {reason})"

print("Chat function and session ready.")

Chat function and session ready.


## Orchestration + guardrail test

Three messages, each run without a session so it starts clean. Prints which specialist agents the orchestrator called:
1. Spending question → `expense_analyst` only
2. Savings question → `expense_analyst -> budget_optimizer`
3. Off-topic → blocked by the input guardrail

In [44]:
from agents import ToolCallItem

tests = [
    "Read data/sample_transactions.csv. What is my current spending?",                # expect: expense_analyst only
    "Read data/sample_transactions.csv and tell me how to spend less on groceries.",  # expect: both
    "Write me a poem about cats",                                                      # expect: blocked
]

for msg in tests:
    print(f">>> {msg}")
    try:
        result = await Runner.run(OrchestratorAgent, msg)  # no session, so each test starts clean
        called = [item.raw_item.name for item in result.new_items if isinstance(item, ToolCallItem)]
        print("Agents called:", " -> ".join(called) or "none")
        print(result.final_output)
    except InputGuardrailTripwireTriggered as e:
        print("Blocked by input guardrail:", e.guardrail_result.output.output_info.reasoning)
    except OutputGuardrailTripwireTriggered as e:
        print("Blocked by output guardrail:", e.guardrail_result.output.output_info.reasoning)
    print("-" * 60)

>>> Read data/sample_transactions.csv. What is my current spending?
Agents called: expense_analyst
Here’s the spending summary for `data/sample_transactions.csv`.

**Tools used**
- **ExpenseAnalystAgent used:** `functions.parse_csv_transactions_tool` to parse and validate the CSV transactions before analysis.

**ExpenseAnalystAgent output: Monthly expense summary by category**

The file contains **25 transactions** for **October 2026**.

### Spending by category
| Category | Total |
|---|---:|
| Rent | $750.00 |
| Groceries | $284.04 |
| Shopping / Household | $187.78 |
| Utilities | $132.34 |
| Transportation | $112.75 |
| Dining / Coffee | $67.90 |
| Entertainment / Subscriptions | $48.98 |
| Education | $36.00 |
| Health / Pharmacy | $22.75 |

### Total spending
**$1,642.54**

### Category notes
- **Largest expense:** Rent at **$750.00**
- **Next highest categories:** Groceries (**$284.04**) and Shopping / Household (**$187.78**)
- **Utilities total:** Electric Bill + Internet Bill 

## 10. Test — PDF

Asks the agent to read the placeholder PDF and summarize expenses. Expect a list of bills plus optimization tips.

In [45]:
result = await Runner.run(
    ExpenseAnalystAgent,
    "Read data/MonthlySpendingData_Placeholder.pdf and summarize my expenses."
)

print(result.final_output)

Here’s your expense summary.

What was provided by the ExpenseAnalystAgent:
- I used the tool `functions.extract_pdf_text` to read `data/MonthlySpendingData_Placeholder.pdf`.
- Based on the extracted text, your monthly expenses are:

Monthly expenses
- Rent: $1,000
- Gas: $200
- Groceries: $400
- Spotify subscription: $10
- Netflix subscription: $15
- Take-out: $50
- Car payment: $160
- Car insurance: $100
- Medical insurance: $300
- Amazon orders: $100
- Miscellaneous: $400

Reported totals
- Total monthly spending: $3,500
- Money taken home: $765

Quick observations
- Your largest expense is rent at $1,000.
- Other major categories are groceries ($400), miscellaneous ($400), and medical insurance ($300).
- Subscriptions are relatively small at $25 total.
- The listed category amounts add up to $2,735, which does not match the reported total monthly spending of $3,500.
- If “money taken home” means remaining money after expenses, that also appears inconsistent with the reported spendi

## 11. Test — CSV

Asks the agent to read `data/sample_transactions.csv`. Expect a transaction count, total, and category breakdown.

In [46]:
result = await Runner.run(
    ExpenseAnalystAgent,
    "Read data/sample_transactions.csv")

print(result.final_output)

**ExpenseAnalystAgent output**

I read `data/sample_transactions.csv` and identified the monthly expenses for **October 2026**.

**Tool used**
- `functions.parse_csv_transactions_tool` — used to parse and validate the CSV transaction file.

### Monthly expenses list
| Date | Description | Amount |
|---|---|---:|
| 2026-10-01 | Walmart Groceries | $85.50 |
| 2026-10-02 | Starbucks Coffee | $6.75 |
| 2026-10-03 | Amazon Household Items | $42.99 |
| 2026-10-04 | Kroger Groceries | $67.24 |
| 2026-10-05 | Shell Gas Station | $38.60 |
| 2026-10-06 | Chipotle Lunch | $13.45 |
| 2026-10-07 | Netflix Subscription | $15.49 |
| 2026-10-08 | Target Shopping | $54.80 |
| 2026-10-09 | Uber Ride | $18.20 |
| 2026-10-10 | University Bookstore | $36.00 |
| 2026-10-11 | McDonalds Dinner | $11.25 |
| 2026-10-12 | Publix Groceries | $73.18 |
| 2026-10-13 | Spotify Premium | $11.99 |
| 2026-10-14 | Electric Bill | $82.35 |
| 2026-10-15 | Apartment Rent | $750.00 |
| 2026-10-16 | Starbucks Coffee | $5.95 |

## 12. Gradio chat

Opens a chat UI in the browser. Session memory goes to `user_expenses.db`. Try: *Read data/sample_transactions.csv* or *summarize my PDF expenses*.

In [47]:
chat_session = SQLiteSession("gradio_user", "user_expenses.db")
gr.ChatInterface(chained_gradio_chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Notes (from Chris)

Skeleton agent: one PDF for now, tools first, memory later. Last cell is teammate comments only.

In [48]:
# Hey guys, so this is basically the skeleton of the Expenses Agent, I formatted it in a way that it will read a single pdf file when given the tools to do so
#We will end up adding memory as well, preferably after the tools are implemented for this agent
#It may be a little thrown together in terms of the structure, but I made what I could on my own. If you guys have issues with it, we will modify it together. 
#-Chris